# Interpolación IDW: mapa de permeabilidad a partir de pozos
### Notebook v2.4 · versión didáctica

**Objetivo:** estimar la permeabilidad $k$ en todo un campo de 90 m × 72 m a partir de 64 pozos, con el método de **distancia inversa ponderada (IDW)**

**Datos:** `perm_1_case1.csv` (misma carpeta).

| Paso | Ecuación | En palabras |
|---|---|---|
| 1. Distancia | $d_i=\sqrt{(x_i-x_0)^2+(y_i-y_0)^2}$ | Qué tan lejos está cada pozo $i$ del nodo $(x_0, y_0)$ |
| 2. Peso | $w_i=1/d_i^{\,p}$ | Pozo cercano → peso grande ($p=2$) |
| 3. Promedio | $\hat k_0=\sum w_i\,k_i \,/\, \sum w_i$ | Promedio de los pozos, pesado por cercanía |

## 🎯 Tu misión
con **64 pozos** se debe de construir el **mapa de $k$** .

| Símbolo | Qué haces |
|---|---|
| ✏️ | **Completa** los `___` antes de ejecutar |
| 🤔 | **Predice** qué va a salir, y luego ejecuta |
| 🔍 | **Comprueba**: ✅ = vas bien, ❌ = revisa la celda anterior |
| 🗺️ | **Interpreta** y abre «Ver respuesta» |

Si ejecutas una celda sin completar o con un error, Python lo dice en la última línea del mensaje rojo. Los tres más comunes:

| Mensaje | Qué significa |
|---|---|
| `NameError: name '___' is not defined` | Falta rellenar un espacio, o escribiste mal el nombre de una variable |
| `FileNotFoundError` | El nombre del archivo está mal, o no está en la misma carpeta |
| `SyntaxError` | Falta una comilla, un paréntesis o los dos puntos `:` |

Corrige y vuelve a ejecutar la misma celda.

## 1. Librerías

In [ ]:
%pip install numpy pandas matplotlib --quiet

In [ ]:
import numpy as np                       # cálculos
import pandas as pd                      # tablas
import matplotlib.pyplot as plt          # gráficas
from matplotlib.colors import LogNorm    # escala de colores logarítmica
print("Librerías cargadas")

## 2. Los datos: ¿qué se conoce?

> | Instrucción | Qué hace |
> |---|---|
> | `variable = valor` | Guarda un valor con un nombre (`=` significa «guardar en») |
> | `pd.read_csv("archivo.csv")` | Lee el archivo como **tabla**; el texto siempre va entre comillas |
> | `datos["k (mD)"]` · `.to_numpy()` | Toma una columna por su nombre y la convierte en **arreglo** (lista de números para calcular) |
> | `len(k)` · `k.min()` · `k.max()` | Cuántos elementos hay, el mínimo y el máximo |
> | `x[0]` | Primer elemento: Python cuenta **desde 0** |

In [ ]:
datos = pd.read_csv("___")                   # ✏️ nombre del archivo

In [ ]:
x = datos["___"].to_numpy()                  # ✏️ columna de la coordenada x
y = datos["___"].to_numpy()                  # ✏️ columna de la coordenada y
k = datos["___"].to_numpy()                  # ✏️ columna de la permeabilidad

In [ ]:
print("Pozos:", ___(k), "  k mín =", round(k.min(), 1), "mD   k máx =", round(k.max(), 1), "mD")   # ✏️ ¿qué función cuenta elementos?
datos.head()                                 # primeras 5 filas

## Visualize data points

In [ ]:
plt.figure(figsize=(8, 6))                                             # lienzo
plt.scatter(___, ___, c=___, cmap="viridis", norm=LogNorm(), s=90, edgecolor="black")   # ✏️ posición (x, y) y color (k)
plt.colorbar(label="k [mD]"); plt.xlabel("x [m]"); plt.ylabel("y [m]")  # etiquetas
plt.title("Pozos y su permeabilidad"); plt.show()                     # título y mostrar

## 3. La malla: ¿dónde se quiere estimar?

La **malla** son puntos ordenados (nodos) que cubren el campo, igual que las celdas de un simulador. En $x$ va de 0 a 89 m y en $y$ de 0 a 71 m, con un nodo cada 1 m: $90\times72=$ **6 480 nodos**.

> 🧩 `np.arange(inicio, fin, paso)` genera números desde `inicio` de `paso` en `paso`, y se detiene **antes** de `fin`. Por ejemplo, `np.arange(0, 10, 2)` da `[0 2 4 6 8]`. Por eso se escribe `final + espaciamiento`: para incluir el último nodo.

In [ ]:
espaciamiento = 1                                                # [m] distancia entre nodos
final_x, final_y = ___, ___                                      # ✏️ último nodo en x y en y [m]
nodos_x = np.arange(0, ___ + espaciamiento, espaciamiento)       # ✏️ 0, 1, ..., 89
nodos_y = np.arange(0, ___ + espaciamiento, espaciamiento)       # ✏️ 0, 1, ..., 71
print("Posiciones en x:", len(nodos_x), "  en y:", len(nodos_y), "  nodos:", len(nodos_x) * len(nodos_y))

In [ ]:
# 🔍 Comprueba
print("✅ Malla de 90 × 72 = 6480 nodos" if (len(nodos_x), len(nodos_y)) == (90, 72) else "❌ Revisa la malla")

> 🧩 **Ciclo `for`:** `for elemento in lista:` repite las líneas con **sangría** (4 espacios) una vez por elemento. Un `for` dentro de otro recorre **todas las combinaciones**: para cada fila ($y$), todas las columnas ($x$).

Para no llenar la figura, se dibuja solo un nodo cada 5 m. Es la misma idea que después se usará para estimar.

In [ ]:
plt.figure(figsize=(8, 6))
for yn in np.arange(0, 72, 5):                         # cada fila (y), cada 5 m
    for xn in np.arange(0, 90, 5):                     #   cada columna (x), cada 5 m
        plt.plot(xn, yn, "+", color="gray")            #     cruz gris en el nodo
plt.scatter(x, y, c="red", edgecolor="black", s=50)    # pozos en rojo
plt.xlabel("x [m]"); plt.ylabel("y [m]"); plt.title("Nodos (cruces, uno cada 5 m) y pozos (rojos)"); plt.show()

## 4. Programar el IDW

> 🧩 **Funciones**
> | Pieza | Qué hace |
> |---|---|
> | `def nombre(entrada1, entrada2):` | **Crea** una función: código con nombre que se puede reutilizar. Su cuerpo lleva sangría |
> | `return resultado` | **Entrega** el resultado |
> | `nombre(45, 36)` | **Usa** la función con esos datos. Definirla no calcula nada; calcula cuando se usa |
>
> **Operaciones con arreglos:** `x - x0` resta $x_0$ a **los 64 pozos a la vez**; `**` es potencia; `np.sqrt` es la raíz; `np.sum` suma todo (el $\Sigma$); `np.maximum(d, 0.001)` cambia distancias de 0 por 0.001 para **no dividir entre cero** cuando un nodo cae sobre un pozo.

In [ ]:
p = ___                                              # ✏️ potencia del IDW

def distancias(x0, y0):                              # paso 1
    return np.___((x - ___)**2 + (y - ___)**2)       # ✏️ d_i = raíz[(x_i - x0)² + (y_i - y0)²]

def idw(x0, y0, p):
    d = np.maximum(distancias(x0, y0), 0.001)        # paso 1 (sin ceros)
    w = 1 / ___**p                                   # ✏️ paso 2: w_i = 1 / d_i^p
    return np.___(w * ___) / np.sum(___)             # ✏️ paso 3: k0 = Σ(w_i·k_i) / Σ(w_i)

✏️ Pistas: compara cada línea con su ecuación. ¿Qué función de NumPy saca la raíz? ¿Cuál suma? ¿Qué variable se eleva a $p$? ¿Qué se multiplica por los pesos en el numerador?

🤔 **Predice:** el pozo 1 está en (15, 0). Calcula a mano su distancia al nodo (45, 36). ¿Qué $k$ debería estimar `idw` justo en (15, 0)?
<details><summary>Ver respuesta</summary>

$d=\sqrt{(15-45)^2+(0-36)^2}=\sqrt{2196}=46.86$ m. En (15, 0) el IDW debe devolver el valor medido del pozo 1 (161.8 mD), porque ese pozo tiene casi todo el peso.
</details>

**Prueba:** `d[:5]` muestra las primeras 5 distancias.

In [ ]:
d = distancias(45, 36)
print("Distancias de los primeros 5 pozos al nodo (45, 36) [m]:", d[:5].round(2))
print("k estimada en (45, 36):", round(idw(45, 36, p), 1), "mD   |   en el pozo 1 (15, 0):", round(idw(15, 0, p), 1), "mD")

## 5. Estimar todos los nodos

| Variable | Qué contendrá | Para qué |
|---|---|---|
| `mapa` | La $k$ estimada de cada nodo en una tabla de 72 filas ($y$) × 90 columnas ($x$) | Dibujar el mapa |
| `lista_x`, `lista_y`, `lista_k` | $x$, $y$ y $k$ estimada de cada nodo (6 480 valores) | Tabla de resultados |

> 🧩 `np.zeros((filas, columnas))` crea una tabla de ceros · `[]` es una lista vacía · `range(72)` da 0, 1, …, 71 (números de fila) · `nodos_x[columna]` toma el elemento de esa posición · `mapa[fila, columna] = k0` escribe en una casilla, como en Excel · `lista.append(v)` agrega `v` al final.

El ciclo recorre la malla **fila por fila**. En cada nodo: coordenadas → `idw` → guardar.

In [ ]:
mapa = np.zeros((len(___), len(___)))              # ✏️ filas = posiciones en y, columnas = posiciones en x
lista_x, lista_y, lista_k = [], [], []             # listas vacías

for fila in range(len(nodos_y)):                   # cada fila (y)
    for columna in range(len(nodos_x)):            #   cada columna (x)
        x0 = nodos_x[___]                          #     ✏️ coordenada x del nodo
        y0 = nodos_y[___]                          #     ✏️ coordenada y del nodo
        k0 = idw(___, ___, p)                      #     ✏️ k estimada
        mapa[___, ___] = k0                        #     ✏️ guardar en mapa (fila, columna)
        lista_x.append(x0); lista_y.append(y0); lista_k.append(___)   # ✏️ ¿qué se guarda en lista_k?

print("Nodos estimados:", len(lista_k), "  nodo (45, 36):", round(mapa[36, 45], 1), "mD",
      "  k mín =", round(mapa.min(), 1), "  k máx =", round(mapa.max(), 1))

✏️ Pistas: `nodos_x` se recorre con el número de **columna** y `nodos_y` con el de **fila**; `mapa` se escribe como `[fila, columna]`.

## 6. Mapa y resultados

> 🧩 `plt.pcolormesh(nodos_x, nodos_y, mapa, ...)` pinta cada nodo con el color de su valor · `{"columna": lista, ...}` es un **diccionario** (nombre → contenido) · `pd.DataFrame({...})` lo convierte en tabla; pandas muestra sus primeras y últimas filas.

In [ ]:
plt.figure(figsize=(9, 6.5))
plt.pcolormesh(nodos_x, nodos_y, ___, cmap="viridis", norm=LogNorm(vmin=k.min(), vmax=k.max()), shading="auto")   # ✏️ ¿qué se dibuja?
plt.colorbar(label="k estimada [mD]"); plt.scatter(x, y, c="white", edgecolor="black", s=30)   # barra y pozos
plt.xlabel("x [m]"); plt.ylabel("y [m]"); plt.title("Mapa de permeabilidad por IDW (p = 2)"); plt.show()

🗺️ 1) ¿Dónde está la mayor permeabilidad? 2) ¿Por qué hay manchas circulares alrededor de algunos pozos? 3) ¿Vería el IDW un canal orientado de norte a sur?
<details><summary>Ver respuestas</summary>

1. Alrededor de (50, 8) y en el borde izquierdo, cerca de (0, 24).
2. Son «ojos de buey»: el peso solo depende de la distancia, así que cada pozo con un valor muy distinto domina en círculo a su alrededor.
3. No. El IDW no usa la dirección. Para reconocer la anisotropía se usan el **semivariograma** y el **Kriging**.
</details>

In [ ]:
resultados = pd.DataFrame({"x [m]": lista_x, "y [m]": lista_y, "k estimada [mD]": np.round(___, 1)})   # ✏️ ¿qué lista va en la columna de k?
print("Verificación en el pozo 1 (15, 0): medida =", round(k[0], 1), "mD   en el mapa =", round(mapa[0, 15], 1), "mD")
resultados

🎉 **Se contruyó un mapa de 6 480 nodos a partir de 64 pozos (puntos).**

🧪 **Reto:** cambia `p` a 1 o a 3 en la sección 4 y vuelve a ejecutar desde ahí hasta el mapa. ¿Se ve más suave o con más «parches»?

## 7. Resumen

| Variable | Contiene |
|---|---|
| `x`, `y`, `k` | Coordenadas [m] y permeabilidad [mD] de los 64 pozos |
| `nodos_x`, `nodos_y` | 90 y 72 posiciones de la malla |
| `p` | Potencia del IDW (2) |
| `distancias`, `idw` | Las dos funciones del método |
| `mapa` | $k$ estimada en forma de malla, 72 filas ($y$) × 90 columnas ($x$) |
| `lista_x`, `lista_y`, `lista_k` · `resultados` | Los 6 480 nodos como listas y como tabla |

| Sintaxis clave | Para qué se usó |
|---|---|
| `import … as` | Cargar librerías |
| `pd.read_csv`, `datos["col"]`, `.to_numpy()` | Leer los datos y separar columnas |
| `np.arange(inicio, fin, paso)` | Crear la malla |
| `def … return`, `np.sqrt`, `**`, `np.sum` | Programar las tres ecuaciones |
| `for` dentro de `for`, `range`, `mapa[fila, columna]`, `.append()` | Recorrer los nodos y guardar |
| `plt.scatter`, `plt.pcolormesh`, `pd.DataFrame` | Mostrar pozos, mapa y tabla |

**Conclusión:** con 64 pozos y tres ecuaciones se estimó la permeabilidad en 6 480 nodos. El IDW es simple y respeta los datos, pero solo usa la distancia: no ve direcciones de continuidad ni mide la incertidumbre. Para eso está el **Kriging**.